In [0]:
%sql
-- ================================================================
-- [DQ-1] KPI CARDS — Pipeline Health Summary (Top Row)
-- Visual: 5 Counter Tiles
-- ================================================================
SELECT
  SUM(passed_records) AS total_clean_records,
  SUM(dropped_records) AS total_quarantined_records,
  SUM(total_records)  AS total_ingested_records,
  ROUND(SUM(dropped_records) / SUM(total_records) * 100, 2) AS overall_drop_rate_pct,
  COUNT(DISTINCT target_table)                      AS tables_audited
FROM maven_catalog.gold_schema.gold_pipeline_health_audit;


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- ================================================================
-- [DQ-2] Quarantine Breakdown by Table (Bar Chart)
-- Visual: Horizontal bar — X: dropped_records, Y: source table
-- ================================================================
SELECT
  target_table,
  silver_source,
  passed_records,
  dropped_records,
  failure_rate_pct,
  _audit_timestamp
FROM maven_catalog.gold_schema.gold_pipeline_health_audit
ORDER BY dropped_records DESC;


In [0]:
%sql

-- ================================================================
-- [DQ-3] Drop Rate % by Pipeline (Gauge / Donut)
-- Visual: Donut chart — each slice = one pipeline's drop rate
-- ================================================================
SELECT
  target_table,
  failure_rate_pct,
  CASE
    WHEN failure_rate_pct = 0   THEN 'Healthy'
    WHEN failure_rate_pct < 2   THEN 'Acceptable'
    WHEN failure_rate_pct < 5   THEN 'Warning'
    ELSE                             'Critical'
  END AS health_status
FROM maven_catalog.gold_schema.gold_pipeline_health_audit
ORDER BY failure_rate_pct DESC;

In [0]:
%sql
-- ================================================================
-- [DQ-4] Quarantine Reasons Breakdown (Top Failure Modes)
-- Visual: Stacked horizontal bar — X: count, Y: source, Color: reason
-- ================================================================
SELECT 'transactions' AS source_table, _quarantine_reasons, COUNT(*) AS row_count
FROM maven_catalog.silver_schema.quarantine_transactions
GROUP BY _quarantine_reasons

UNION ALL

SELECT 'returns', _quarantine_reasons, COUNT(*)
FROM maven_catalog.silver_schema.quarantine_returns
GROUP BY _quarantine_reasons

UNION ALL

SELECT 'products', _quarantine_reasons, COUNT(*)
FROM maven_catalog.silver_schema.quarantine_products
GROUP BY _quarantine_reasons

UNION ALL

SELECT 'kafka_orders', _quarantine_reasons, COUNT(*)
FROM maven_catalog.silver_schema.quarantine_kafka_orders
GROUP BY _quarantine_reasons

UNION ALL

SELECT 'kafka_inventory', _quarantine_reasons, COUNT(*)
FROM maven_catalog.silver_schema.quarantine_kafka_inventory
GROUP BY _quarantine_reasons

ORDER BY row_count DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- ================================================================
-- [DQ-5] Records Per Layer — Bronze vs Silver vs Gold (Flow Chart)
-- Visual: Grouped bar chart — shows record flow + drops between layers
-- ================================================================
SELECT 'Bronze' AS layer, 'transactions' AS pipeline, COUNT(*) AS row_count
FROM maven_catalog.bronze_schema.brz_transactions
UNION ALL
SELECT 'Silver', 'transactions', COUNT(*)
FROM maven_catalog.silver_schema.slv_transactions
UNION ALL
SELECT 'Quarantine', 'transactions', COUNT(*)
FROM maven_catalog.silver_schema.quarantine_transactions
UNION ALL
SELECT 'Gold', 'transactions', COUNT(*)
FROM maven_catalog.gold_schema.fact_sales

UNION ALL

SELECT 'Bronze', 'returns', COUNT(*)
FROM maven_catalog.bronze_schema.brz_returns
UNION ALL
SELECT 'Silver', 'returns', COUNT(*)
FROM maven_catalog.silver_schema.slv_returns
UNION ALL
SELECT 'Quarantine', 'returns', COUNT(*)
FROM maven_catalog.silver_schema.quarantine_returns
UNION ALL
SELECT 'Gold', 'returns', COUNT(*)
FROM maven_catalog.gold_schema.fact_returns

UNION ALL

SELECT 'Bronze', 'products', COUNT(*)
FROM maven_catalog.bronze_schema.brz_products_mongo_dlt
UNION ALL
SELECT 'Silver', 'products', COUNT(*)
FROM maven_catalog.silver_schema.slv_products
UNION ALL
SELECT 'Quarantine', 'products', COUNT(*)
FROM maven_catalog.silver_schema.quarantine_products
UNION ALL
SELECT 'Gold', 'products', COUNT(*)
FROM maven_catalog.gold_schema.dim_products

ORDER BY pipeline, layer;


In [0]:
%sql
-- ================================================================
-- [DQ-7] DLT Pipeline Event Log (Latest Runs)
-- Visual: Table showing last 50 pipeline events
-- NOTE: Replace TABLE() reference with your actual DLT pipeline ID
-- ================================================================
SELECT
  timestamp,
  details:flow_name::STRING                              AS flow_name,
  details:flow_progress.metrics.num_output_rows::BIGINT  AS records_processed,
  details:flow_progress.status::STRING                   AS status,
  event_type
FROM event_log(TABLE(maven_catalog.silver_schema.slv_transactions))
WHERE event_type IN ('flow_progress', 'flow_definition')
ORDER BY timestamp DESC
LIMIT 50;

In [0]:
%sql
-- ================================================================
-- [DQ-8] Drop Rate Trend Over Time (Line Chart)
-- Visual: Line chart — X: audit_timestamp, Y: failure_rate_pct, Color: target_table
-- NOTE: This requires multiple pipeline runs to show trend.
--       The gold_pipeline_health_audit appends on each run.
-- ================================================================
SELECT
  DATE_TRUNC('hour', _audit_timestamp) AS audit_hour,
  target_table,
  failure_rate_pct,
  total_records,
  dropped_records
FROM maven_catalog.gold_schema.gold_pipeline_health_audit
ORDER BY audit_hour DESC, target_table;
